# 01 - EDA

Notebook này tách phần phân tích khám phá dữ liệu từ `ML_Project.ipynb` và có thể chạy độc lập.

## Bước 1: Phân tích khám phá dữ liệu (EDA)

Chúng ta sẽ tiến hành phân tích khám phá dữ liệu để hiểu rõ cấu trúc, phân bố và mối quan hệ giữa các biến số trong tập dữ liệu. Mục tiêu là phát hiện các đặc điểm quan trọng, nhận diện vấn đề (như giá trị thiếu, ngoại lai, độ lệch) và đưa ra các quyết định xử lý sơ bộ cho quá trình xây dựng mô hình ML.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

DATA_PATH = Path('../data/housing.csv.zip')
FIGURES_DIR = Path('../../docs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
df = pd.read_csv(DATA_PATH)

display(df.head())
print('Shape:', df.shape)
print('\nInfo:')
display(df.info())
print('\nDescribe:')
display(df.describe())
print('\nMissing values:')
display(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print(f"Figures directory: {FIGURES_DIR}")

### Hình 1: Phân bố biến mục tiêu (`median_house_value`)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['median_house_value'], kde=True, bins=50)
plt.title('Phân bố giá nhà trung bình (Median House Value)')
plt.xlabel('Giá nhà trung bình (USD)')
plt.ylabel('Số lượng')
plt.savefig(FIGURES_DIR / 'plot_1_median_house_value_distribution.png', bbox_inches='tight')
plt.show()

**1. Hình cho thấy gì?**
Biểu đồ phân bố của `median_house_value` cho thấy phần lớn giá nhà tập trung ở mức thấp và trung bình, với một đỉnh rõ ràng xung quanh giá trị 150,000 USD. Có một số lượng đáng kể các ngôi nhà có giá trị bị 'cắt' ở mức 500,000 USD (giá trị tối đa). Điều này có thể cho thấy một ngưỡng trần trong dữ liệu hoặc cách dữ liệu được thu thập.

**2. Ý nghĩa với bài toán?**
Sự phân bố lệch phải (right-skewed) của biến mục tiêu là phổ biến trong dữ liệu giá nhà. Giá trị bị cắt ở mức tối đa (500,000 USD) là một vấn đề nghiêm trọng, vì mô hình sẽ không thể dự đoán giá trị vượt quá ngưỡng này. Điều này có thể dẫn đến việc đánh giá thấp giá trị thực của những ngôi nhà đắt tiền.

**3. Quyết định xử lý tiếp theo?**
Cần xem xét loại bỏ hoặc xử lý các quan sát có giá nhà trung bình ở mức 500,000 USD nếu chúng chiếm tỷ lệ nhỏ và không đại diện cho phân bố tự nhiên của giá nhà. Nếu chúng đại diện cho một phần quan trọng của dữ liệu, có thể cần cân nhắc các kỹ thuật biến đổi logarit cho biến mục tiêu để giảm độ lệch hoặc sử dụng các mô hình có thể xử lý tốt hơn dữ liệu lệch.

### Hình 2: Phân bố các đặc trưng số quan trọng (`median_income`, `housing_median_age`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(df['median_income'], kde=True, bins=50, ax=axes[0])
axes[0].set_title('Phân bố thu nhập trung bình (Median Income)')
axes[0].set_xlabel('Thu nhập trung bình (x10,000 USD)')
axes[0].set_ylabel('Số lượng')

sns.histplot(df['housing_median_age'], kde=True, bins=50, ax=axes[1])
axes[1].set_title('Phân bố tuổi nhà trung bình (Housing Median Age)')
axes[1].set_xlabel('Tuổi nhà trung bình')
axes[1].set_ylabel('Số lượng')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'plot_2_numerical_features_distribution.png', bbox_inches='tight')
plt.show()

**1. Hình cho thấy gì?**
*   **`median_income`**: Phân bố có vẻ lệch phải (right-skewed), với phần lớn thu nhập trung bình tập trung ở mức thấp hơn và có một đuôi dài về phía thu nhập cao hơn. Cũng có một vài giá trị bị cắt ở mức tối đa (khoảng 15) tương tự như `median_house_value`.
*   **`housing_median_age`**: Phân bố có vẻ đồng đều hơn một chút, nhưng có một đỉnh ở tuổi 52, cho thấy có nhiều ngôi nhà cũ trong tập dữ liệu. Cũng có một số giá trị bị cắt ở mức tuổi nhà tối đa (52). Điều này cũng có thể là một ngưỡng trần trong dữ liệu.

**2. Ý nghĩa với bài toán?**
*   **`median_income`**: Là một yếu tố dự đoán mạnh mẽ cho giá nhà. Dạng phân bố lệch có thể ảnh hưởng đến hiệu suất của một số mô hình ML. Giá trị bị cắt có thể làm sai lệch mối quan hệ giữa thu nhập và giá nhà.
*   **`housing_median_age`**: Sự tập trung ở tuổi 52 có thể là một giá trị ngoại lai hoặc một ngưỡng trần. Điều này cần được điều tra thêm vì tuổi nhà có thể là một đặc trưng quan trọng.

**3. Quyết định xử lý tiếp theo?**
*   **`median_income`**: Cần xem xét biến đổi logarit hoặc các biến đổi khác để làm cho phân bố gần với phân bố chuẩn hơn, giúp cải thiện hiệu suất của mô hình. Các giá trị bị cắt có thể được xử lý tương tự như `median_house_value`.
*   **`housing_median_age`**: Kiểm tra giá trị 52 là một giá trị thực hay là một ngưỡng dữ liệu. Nếu là ngưỡng, cần lưu ý khi xây dựng mô hình hoặc xem xét việc mã hóa biến này.

### Hình 3: Bản đồ dữ liệu thiếu (Heatmap của `df.isnull()`)

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Bản đồ dữ liệu thiếu (Missing Data Heatmap)')
plt.xlabel('Tên cột')
plt.ylabel('Hàng dữ liệu')
plt.savefig(FIGURES_DIR / 'plot_3_missing_data_heatmap.png', bbox_inches='tight')
plt.show()

**1. Hình cho thấy gì?**
Bản đồ nhiệt cho thấy một cột duy nhất là `total_bedrooms` có các giá trị bị thiếu (hiển thị bằng các đường màu khác trên bản đồ). Các cột khác dường như không có dữ liệu thiếu. Số lượng giá trị thiếu trong `total_bedrooms` không quá lớn, nhưng chúng xuất hiện rải rác trên toàn bộ tập dữ liệu.

**2. Ý nghĩa với bài toán?**
Các giá trị thiếu trong `total_bedrooms` cần được xử lý trước khi huấn luyện mô hình. Việc bỏ qua các hàng có giá trị thiếu có thể làm mất thông tin quan trọng. `total_bedrooms` là một đặc trưng số quan trọng có thể ảnh hưởng đến giá nhà.

**3. Quyết định xử lý tiếp theo?**
Cần điền các giá trị thiếu cho cột `total_bedrooms`. Các phương pháp phổ biến có thể bao gồm: điền bằng giá trị trung bình (`mean`), trung vị (`median`), hoặc giá trị mode. Đối với dữ liệu số và phân bố có thể có ngoại lai, điền bằng giá trị trung vị thường là lựa chọn an toàn hơn.

### Hình 4: Heatmap tương quan (Correlation heatmap) của tất cả các biến số

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Bản đồ nhiệt tương quan giữa các biến số')
plt.savefig(FIGURES_DIR / 'plot_4_correlation_heatmap.png', bbox_inches='tight')
plt.show()

**1. Hình cho thấy gì?**
Bản đồ nhiệt tương quan cho thấy:
*   **`median_income`** có mối tương quan mạnh nhất và dương với **`median_house_value`** (0.69). Điều này có nghĩa là thu nhập trung bình càng cao thì giá nhà trung bình càng cao.
*   **`total_rooms`**, **`households`**, **`total_bedrooms`**, và **`population`** có mối tương quan dương vừa phải với `median_house_value`.
*   **`latitude`** và **`longitude`** có tương quan âm đáng kể với `median_house_value`, cho thấy vị trí địa lý đóng vai trò quan trọng.
*   Có mối tương quan cao giữa các biến như `total_rooms`, `total_bedrooms`, `population`, `households` (ví dụ: `total_rooms` và `total_bedrooms` có tương quan 0.93), điều này cho thấy có thể tồn tại đa cộng tuyến.

**2. Ý nghĩa với bài toán?**
*   **`median_income`** là đặc trưng quan trọng nhất để dự đoán giá nhà.
*   Sự tương quan cao giữa các biến đếm (như `total_rooms`, `total_bedrooms`, `population`, `households`) có thể gây ra vấn đề đa cộng tuyến trong các mô hình hồi quy tuyến tính, làm giảm độ ổn định của các hệ số. Điều này cần được xử lý.
*   `ocean_proximity` là một biến phân loại, không xuất hiện trong heatmap này nhưng cũng có thể là một yếu tố quan trọng.

**3. Quyết định xử lý tiếp theo?**
*   Giữ lại **`median_income`** làm đặc trưng chính.
*   Đối với các biến có đa cộng tuyến cao, cân nhắc tạo ra các đặc trưng mới từ chúng (ví dụ: `rooms_per_household`, `bedrooms_per_room`, `population_per_household`) hoặc sử dụng các kỹ thuật giảm chiều như PCA, hoặc chỉ chọn một trong số các biến có tương quan cao nếu chúng thể hiện cùng một thông tin.

### Hình 5: Mối quan hệ giữa đặc trưng chính (`median_income`) và biến mục tiêu (`median_house_value`)

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(x='median_income', y='median_house_value', data=df, alpha=0.4)
plt.title('Mối quan hệ giữa thu nhập trung bình và giá nhà trung bình')
plt.xlabel('Thu nhập trung bình (x10,000 USD)')
plt.ylabel('Giá nhà trung bình (USD)')
plt.savefig(FIGURES_DIR / 'plot_5_median_income_vs_house_value.png', bbox_inches='tight')
plt.show()

**1. Hình cho thấy gì?**
Biểu đồ phân tán cho thấy mối quan hệ tương quan tuyến tính dương mạnh mẽ giữa `median_income` và `median_house_value`. Khi thu nhập trung bình tăng lên, giá nhà trung bình cũng có xu hướng tăng. Tuy nhiên, vẫn có thể thấy rõ các đường ngang ở các mức giá nhà nhất định, đặc biệt là ở mức 500,000 USD, khẳng định lại vấn đề dữ liệu bị cắt trần.

**2. Ý nghĩa với bài toán?**
`median_income` là một đặc trưng cực kỳ quan trọng và có khả năng dự đoán cao đối với `median_house_value`. Mối quan hệ tuyến tính này rất tốt cho các mô hình hồi quy. Tuy nhiên, vấn đề dữ liệu bị cắt trần ở `median_house_value` có thể làm suy yếu khả năng dự đoán của mô hình ở phân khúc giá cao.

**3. Quyết định xử lý tiếp theo?**
*   Sử dụng `median_income` làm một trong những đặc trưng chính.
*   Cần xử lý vấn đề `median_house_value` bị cắt trần. Tùy thuộc vào mục tiêu, có thể loại bỏ các mẫu này hoặc biến đổi biến mục tiêu để giảm thiểu ảnh hưởng của điểm dữ liệu này.

### Hình 6: Scatterplot matrix (Pairplot) cho top 4 đặc trưng quan trọng nhất

In [ ]:
important_features = ['median_income', 'housing_median_age', 'total_rooms', 'median_house_value']

# Đối với pairplot, cần tránh các cột có quá nhiều giá trị bị thiếu nếu không đã xử lý.
# Ở đây, 'total_bedrooms' có thiếu, nên chúng ta sẽ bỏ qua nó trong pairplot để tránh lỗi hoặc kết quả không mong muốn nếu chưa xử lý thiếu.

# Lọc DataFrame chỉ với các cột quan trọng đã chọn
df_subset = df[important_features]

# Tạo pairplot
sns.pairplot(df_subset)
plt.suptitle('Pairplot của các đặc trưng quan trọng', y=1.02) # y=1.02 để tiêu đề không bị chồng lên biểu đồ
plt.savefig(FIGURES_DIR / 'plot_6_pairplot_important_features.png', bbox_inches='tight')
plt.show()

**1. Hình cho thấy gì?**
Biểu đồ `pairplot` cung cấp cái nhìn tổng quan về mối quan hệ cặp đôi giữa 4 đặc trưng đã chọn (`median_income`, `housing_median_age`, `total_rooms`, `median_house_value`) và phân bố của từng đặc trưng.
*   Các biểu đồ histogram trên đường chéo chính cho thấy phân bố của từng biến. `median_house_value` và `median_income` có vẻ lệch phải.
*   Biểu đồ phân tán (`scatterplot`) ngoài đường chéo chính thể hiện mối quan hệ giữa các cặp biến.
*   Mối quan hệ tuyến tính giữa `median_income` và `median_house_value` là rõ ràng nhất.
*   `total_rooms` có mối quan hệ tích cực với `median_house_value`, nhưng cũng có sự phân cụm rõ rệt.
*   `housing_median_age` có vẻ có ít mối quan hệ tuyến tính trực tiếp với các biến khác, mặc dù có một số pattern phân tán nhất định.

**2. Ý nghĩa với bài toán?**
*   `Pairplot` xác nhận lại `median_income` là đặc trưng có mối quan hệ rõ ràng nhất với biến mục tiêu.
*   Sự phân tán dữ liệu và các pattern phi tuyến giữa các đặc trưng có thể gợi ý rằng các mô hình tuyến tính đơn giản có thể không nắm bắt được tất cả các mối quan hệ. Các thuật toán như cây quyết định, rừng ngẫu nhiên, hoặc gradient boosting có thể phù hợp hơn.
*   Các điểm ngoại lai và sự phân cụm trong biểu đồ phân tán cần được xem xét để đảm bảo chúng không làm sai lệch mô hình.

**3. Quyết định xử lý tiếp theo?**
*   Xem xét các đặc trưng phi tuyến tính hoặc tạo ra các đặc trưng tương tác nếu cần thiết.
*   Sử dụng các mô hình Machine Learning mạnh mẽ có khả năng xử lý các mối quan hệ phi tuyến và các tương tác giữa các đặc trưng.
*   Tiếp tục làm sạch dữ liệu, bao gồm xử lý các giá trị ngoại lai nếu chúng ảnh hưởng tiêu cực đến hiệu suất mô hình.